# XRD Pipeline — Level 1: Phase Identification

**Workflow**: Load pattern → Preprocess → Fetch candidates (Materials Project) → Score/rank → **CP1: human selects phase set** → save for Rietveld (notebook 02)

All code is self-contained in this notebook. No external local modules required.

**Dependencies** (install once):
```
pip install numpy scipy matplotlib pymatgen mp-api tqdm ipywidgets dara-xrd
```
BGMN binary (for DARA) and GSAS-II (for Rietveld) require manual install — see `INSTALL.md`.

---
## 0. Imports

In [ ]:
import os, re, json, pickle, hashlib, tempfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter
from tqdm.notebook import tqdm

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

---
## 1. Configuration

**Edit this cell for your sample.**

In [ ]:
# ── Edit these ──────────────────────────────────────────────────────────────
XRD_FILE            = Path("../data/sample.xye")    # path to .xy or .xye file
ELEMENTS            = ["Li", "Ni", "Mn", "Co", "O"] # chemical system
WAVELENGTH          = 1.56                         # 'CuKa', 'MoKa', or numeric Å string
MP_API_KEY          = None                           # or paste key; reads MP_API_KEY env var
BGMN_PATH           = None                           # Path(r"C:\bgmn\bgmn.exe"); None = pymatgen
TT_MIN, TT_MAX      = 10.0, 80.0                    # 2θ range to analyse (degrees)
E_HULL_MAX          = 0.1                            # energy above hull filter (eV/atom)
MAX_CANDIDATES      = 200
CACHE_DIR           = Path(".cache")
# ────────────────────────────────────────────────────────────────────────────

CACHE_DIR.mkdir(exist_ok=True)
MP_API_KEY = MP_API_KEY or os.environ.get("MP_API_KEY")

_WAVELENGTHS = {"CuKa": 1.5406, "CuKa1": 1.5406, "CuKa2": 1.5444,
                "MoKa": 0.7107, "CoKa": 1.7902, "CrKa": 2.2897, "FeKa": 1.9373}
WAVELENGTH_A = _WAVELENGTHS.get(WAVELENGTH, float(WAVELENGTH))

bgmn_available = BGMN_PATH is not None and Path(BGMN_PATH).exists()
print(f"Chemical system : {ELEMENTS}")
print(f"Wavelength      : {WAVELENGTH}  ({WAVELENGTH_A:.4f} Å)")
print(f"DARA/BGMN       : {'YES — ' + str(BGMN_PATH) if bgmn_available else 'not configured — using pymatgen scorer'}")
print(f"MP API key      : {'set' if MP_API_KEY else 'NOT SET — fetch will fail'}")

---
## 2. XRD File Parser

In [ ]:
@dataclass
class XRDPattern:
    two_theta: np.ndarray
    intensity: np.ndarray
    sigma:     np.ndarray
    source_file: Optional[Path] = None
    wavelength_a: float = 1.5406
    metadata: dict = field(default_factory=dict)

    @property
    def step_size(self):
        return float(np.mean(np.diff(self.two_theta)))

    @property
    def angular_range(self):
        return float(self.two_theta[0]), float(self.two_theta[-1])

    def trim(self, lo, hi):
        m = (self.two_theta >= lo) & (self.two_theta <= hi)
        return XRDPattern(self.two_theta[m], self.intensity[m], self.sigma[m],
                          self.source_file, self.wavelength_a, self.metadata.copy())


def load_xrd(path, wavelength_a=1.5406, tt_min=None, tt_max=None):
    """Parse a .xy or .xye text file → XRDPattern."""
    path = Path(path)
    rows, meta = [], {}
    _num = re.compile(r"^[\s\t]*[-+]?\d")
    with open(path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            s = line.strip()
            if not s or s.startswith("#") or not _num.match(s):
                if s:
                    meta.setdefault("header", []).append(s.lstrip("#").strip())
                continue
            try:
                rows.append([float(x) for x in s.split()])
            except ValueError:
                pass
    if not rows:
        raise ValueError(f"No numeric data found in {path}")
    arr = np.array(rows)
    tt, I = arr[:, 0], arr[:, 1]
    sig = arr[:, 2] if arr.shape[1] >= 3 else np.sqrt(np.maximum(I, 1.0))
    if not np.all(np.diff(tt) > 0):
        o = np.argsort(tt); tt, I, sig = tt[o], I[o], sig[o]
    p = XRDPattern(tt, I, sig, path, wavelength_a, meta)
    if tt_min is not None or tt_max is not None:
        p = p.trim(tt_min or tt[0], tt_max or tt[-1])
    # QC
    lo, hi = p.angular_range
    if hi - lo < 20:
        print(f"[QC WARNING] Angular range only {hi-lo:.1f}° — phase ID may be unreliable")
    if p.step_size > 0.05:
        print(f"[QC WARNING] Step size {p.step_size:.4f}° is coarse")
    return p

---
## 3. Load & Inspect Raw Pattern

In [ ]:
raw = load_xrd(XRD_FILE, wavelength_a=WAVELENGTH_A, tt_min=TT_MIN, tt_max=TT_MAX)
print(f"Loaded : {len(raw.two_theta)} points  |  range {raw.angular_range[0]:.2f}–{raw.angular_range[1]:.2f}°  |  step {raw.step_size:.4f}°")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(raw.two_theta, raw.intensity, lw=0.8, color='k')
ax.set_xlabel("2θ (degrees)"); ax.set_ylabel("Intensity")
ax.set_title(f"Raw pattern — {XRD_FILE.name}")
plt.tight_layout(); plt.show()

---
## 4. Preprocessing Functions

- **SNIP** background subtraction (Ryan et al., NIM A 396, 1997)
- **Rachinger** Kα2 stripping for Cu radiation

In [ ]:
def snip_background(I, n_iter=24):
    """SNIP background: double-log transform → iterative clipping → inverse transform."""
    y = np.maximum(I, 0.0)
    z = np.log(np.log(np.sqrt(y + 1.0) + 1.0) + 1.0)
    n = len(z)
    for w in range(n_iter, 0, -1):
        z_new = z.copy()
        avg = (z[:n - 2*w] + z[2*w:]) / 2.0
        z_new[w:n-w] = np.minimum(z[w:n-w], avg)
        z = z_new
    bg = (np.exp(np.exp(z) - 1.0) - 1.0) ** 2 - 1.0
    return np.maximum(bg, 0.0)


def strip_kalpha2(pattern, ratio=0.5, wl_ka1=1.54056, wl_ka2=1.54439):
    """Rachinger Kα2 correction. Only for Cu (or similar dual-line) sources."""
    tt, I = pattern.two_theta, pattern.intensity.copy().astype(float)
    sin_t = np.sin(np.deg2rad(tt / 2.0))
    sin_ka2 = np.clip(sin_t * (wl_ka2 / wl_ka1), 0, 1)
    delta = np.rad2deg(2.0 * np.arcsin(sin_ka2)) - tt
    step = pattern.step_size
    for i in range(len(tt)):
        k = int(round(delta[i] / step))
        if k > 0 and (i - k) >= 0:
            I[i] = max(I[i] - ratio * I[i - k], 0.0)
    return XRDPattern(tt, I, pattern.sigma.copy(), pattern.source_file,
                      wl_ka1, {**pattern.metadata, "ka2_stripped": True})


def preprocess(pattern, do_bg=True, do_ka2=True, n_iter=24):
    """Apply SNIP background subtraction then Kα2 strip."""
    p = pattern
    bg = None
    if do_bg:
        bg = snip_background(p.intensity, n_iter)
        net = np.maximum(p.intensity - bg, 0.0)
        new_sig = np.sqrt(p.sigma**2 + np.sqrt(np.maximum(bg, 1.0))**2)
        p = XRDPattern(p.two_theta, net, new_sig, p.source_file,
                       p.wavelength_a, {**p.metadata, "background": bg})
    if do_ka2:
        p = strip_kalpha2(p)
    return p, bg

---
## 5. Preprocess & Inspect

In [ ]:
processed, bg = preprocess(raw, do_bg=True, do_ka2=True)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(raw.two_theta, raw.intensity, lw=0.8, color='k', label='raw')
if bg is not None:
    axes[0].plot(raw.two_theta, bg, lw=1.2, color='r', ls='--', label='SNIP background')
axes[0].legend(fontsize=8); axes[0].set_ylabel('Intensity')
axes[0].set_title('Background estimation (SNIP)')

axes[1].plot(processed.two_theta, processed.intensity, lw=0.8, color='steelblue')
axes[1].set_xlabel('2θ (degrees)'); axes[1].set_ylabel('Net intensity')
axes[1].set_title('After background subtraction + Kα2 strip')

plt.tight_layout(); plt.show()

---
## 6. Fetch Phase Candidates from Materials Project

Results are cached locally — subsequent runs are instant.

In [ ]:
@dataclass
class PhaseCandidate:
    mp_id: str
    formula: str
    space_group: str
    space_group_number: int
    energy_above_hull: float
    structure: object
    simulated_pattern: Optional[object] = None
    score: Optional[float] = None
    metadata: dict = field(default_factory=dict)


def fetch_candidates(elements, mp_api_key, e_hull_max=0.1, max_n=200,
                     cache_dir=Path(".cache"), force=False):
    """Query Materials Project for stable phases in the given chemical system."""
    tag = hashlib.md5(("-".join(sorted(elements)) + f"_{e_hull_max:.3f}").encode()).hexdigest()[:12]
    cache_file = cache_dir / f"mp_candidates_{tag}.pkl"

    if not force and cache_file.exists():
        print(f"[cache] Loading candidates from {cache_file.name}")
        with open(cache_file, "rb") as fh:
            return pickle.load(fh)

    print(f"[MP] Querying Materials Project for {elements} ...")
    from mp_api.client import MPRester
    with MPRester(mp_api_key) as mpr:
        docs = mpr.materials.summary.search(
            elements=elements,
            energy_above_hull=(0, e_hull_max),
            fields=["material_id", "formula_pretty", "symmetry",
                    "energy_above_hull", "structure"],
        )

    seen = {}
    for doc in docs:
        try:
            sym = doc.symmetry
            c = PhaseCandidate(
                mp_id=str(doc.material_id),
                formula=doc.formula_pretty,
                space_group=sym.symbol if sym else "?",
                space_group_number=sym.number if sym else 0,
                energy_above_hull=float(doc.energy_above_hull),
                structure=doc.structure,
            )
            key = (c.formula, c.space_group_number)
            if key not in seen or c.energy_above_hull < seen[key].energy_above_hull:
                seen[key] = c
        except Exception as e:
            print(f"  skip {getattr(doc, 'material_id', '?')}: {e}")

    result = sorted(seen.values(), key=lambda c: c.energy_above_hull)[:max_n]
    with open(cache_file, "wb") as fh:
        pickle.dump(result, fh)
    print(f"[cache] Saved {len(result)} candidates → {cache_file.name}")
    return result

In [ ]:
candidates = fetch_candidates(
    elements=ELEMENTS,
    mp_api_key=MP_API_KEY,
    e_hull_max=E_HULL_MAX,
    max_n=MAX_CANDIDATES,
    cache_dir=CACHE_DIR,
)
print(f"\n{len(candidates)} candidates fetched")
print(f"\n{'MP-ID':<15} {'Formula':<30} {'SG#':>4}  {'Ehull (eV/at)':>14}")
print("-"*70)
for c in candidates[:10]:
    print(f"{c.mp_id:<15} {c.formula:<30} {c.space_group_number:>4}  {c.energy_above_hull:>14.4f}")

---
## 7. Simulate XRD Patterns (pymatgen)

Generates theoretical XRD peak positions and intensities for each candidate.  
Results are cached per (phase, wavelength).

In [ ]:
def simulate_patterns(candidates, wavelength_a, tt_min, tt_max, cache_dir, force=False):
    """Simulate XRD pattern for each candidate using pymatgen XRDCalculator."""
    from pymatgen.analysis.diffraction.xrd import XRDCalculator
    calc = XRDCalculator(wavelength=wavelength_a, symprec=0.1)

    for c in tqdm(candidates, desc="Simulating patterns"):
        tag = hashlib.md5(f"{c.mp_id}_{wavelength_a:.4f}".encode()).hexdigest()[:10]
        cf = cache_dir / f"sim_{tag}.pkl"
        if not force and cf.exists():
            with open(cf, "rb") as fh:
                c.simulated_pattern = pickle.load(fh)
            continue
        try:
            pat = calc.get_pattern(c.structure, two_theta_range=(tt_min, tt_max))
            c.simulated_pattern = pat
            with open(cf, "wb") as fh:
                pickle.dump(pat, fh)
        except Exception as e:
            c.simulated_pattern = None
    return candidates


def broadened_pattern(candidate, tt_grid, sigma=0.05):
    """Gaussian-broadened simulation on the experimental 2θ grid (for overlay plots)."""
    if candidate.simulated_pattern is None:
        return np.zeros_like(tt_grid)
    sim_tt = np.array(candidate.simulated_pattern.x)
    sim_I  = np.array(candidate.simulated_pattern.y)
    if sim_I.max() > 0: sim_I /= sim_I.max()
    out = np.zeros_like(tt_grid, dtype=float)
    for t, i in zip(sim_tt, sim_I):
        out += i * np.exp(-0.5 * ((tt_grid - t) / sigma) ** 2)
    return out

In [ ]:
candidates = simulate_patterns(candidates, WAVELENGTH_A, TT_MIN, TT_MAX, CACHE_DIR)

---
## 8. Score & Rank Candidates

Uses **DARA** (if BGMN installed) or **peak-matching** (pymatgen fallback).

In [ ]:
def extract_peaks(pattern, prominence=0.02, min_height=0.01):
    """Find peaks in the processed pattern. Returns (2θ positions, normalised intensities)."""
    I = pattern.intensity
    I_max = I.max()
    if I_max == 0: return np.array([]), np.array([])
    I_n = I / I_max
    idx, _ = find_peaks(I_n, prominence=prominence, height=min_height)
    return pattern.two_theta[idx], I_n[idx]


def peak_match_score(exp_tt, exp_I, sim_tt, sim_I, tol=0.3):
    """Weighted peak-match score ∈ [0,1]. Higher = better."""
    if len(sim_tt) == 0: return 0.0
    if sim_I.max() > 0: sim_I = sim_I / sim_I.max()
    if len(exp_I) > 0 and exp_I.max() > 0: exp_I = exp_I / exp_I.max()
    total_sim = sim_I.sum()
    if total_sim == 0: return 0.0
    matched_w, matched_exp = 0.0, np.zeros(len(exp_tt), dtype=bool)
    for tt_s, I_s in zip(sim_tt, sim_I):
        if len(exp_tt) == 0: break
        j = np.argmin(np.abs(exp_tt - tt_s))
        if abs(exp_tt[j] - tt_s) <= tol:
            matched_w += I_s; matched_exp[j] = True
    unmatched_penalty = 0.3 * (exp_I[~matched_exp].sum() / (exp_I.sum() + 1e-9))
    return float(np.clip(matched_w / total_sim - unmatched_penalty, 0, 1))


def score_candidates_pymatgen(pattern, candidates, tol=0.3):
    exp_tt, exp_I = extract_peaks(pattern)
    print(f"[score] {len(exp_tt)} experimental peaks detected")
    for c in candidates:
        if c.simulated_pattern is None:
            c.score = 0.0; continue
        c.score = peak_match_score(
            exp_tt, exp_I.copy(),
            np.array(c.simulated_pattern.x), np.array(c.simulated_pattern.y), tol
        )
    return sorted(candidates, key=lambda c: c.score or 0, reverse=True)


def score_candidates(pattern, candidates, bgmn_path=None, elements=None):
    if bgmn_path and Path(bgmn_path).exists():
        print("[score] BGMN found — using DARA scorer")
        return _score_dara(pattern, candidates, bgmn_path, elements)
    print("[score] Using pymatgen peak-matching scorer")
    return score_candidates_pymatgen(pattern, candidates)


def _score_dara(pattern, candidates, bgmn_path, elements):
    try:
        from dara import DARA as DARASolver
        work = Path(tempfile.mkdtemp(prefix="dara_"))
        xye = work / "pattern.xye"
        np.savetxt(xye, np.column_stack([pattern.two_theta, pattern.intensity, pattern.sigma]),
                   fmt="%.6f", header="2theta Intensity Sigma", comments="# ")
        solver = DARASolver(bgmn_executable=str(bgmn_path), working_dir=str(work))
        results = solver.identify(pattern_file=str(xye), elements=elements)
        id_map = {c.mp_id: c for c in candidates}
        for res in results:
            rwp = res.get("rwp", 100.0)
            s = max(0.0, 1.0 - rwp / 100.0)
            for pid in res.get("phase_ids", []):
                if pid in id_map and (id_map[pid].score is None or s > id_map[pid].score):
                    id_map[pid].score = s
                    id_map[pid].metadata["rwp"] = rwp
        return sorted(candidates, key=lambda c: c.score or 0, reverse=True)
    except Exception as exc:
        print(f"[DARA] Error: {exc} — falling back to pymatgen scorer")
        return score_candidates_pymatgen(pattern, candidates)

In [ ]:
ranked = score_candidates(processed, candidates, bgmn_path=BGMN_PATH, elements=ELEMENTS)

print(f"\n{'Rank':<5} {'MP-ID':<15} {'Formula':<30} {'SG#':>4}  {'Ehull':>7}  {'Score':>7}")
print("-"*75)
for i, c in enumerate(ranked[:15]):
    print(f"{i+1:<5} {c.mp_id:<15} {c.formula:<30} {c.space_group_number:>4}  {c.energy_above_hull:>7.4f}  {c.score:>7.4f}")

---
## 9. Visualise Top Candidates

In [ ]:
TOP_N = 5
top = [c for c in ranked[:TOP_N] if c.simulated_pattern is not None]

exp_tt = processed.two_theta
exp_I  = processed.intensity / (processed.intensity.max() or 1)
cmap   = plt.cm.tab10

fig, axes = plt.subplots(len(top), 1, figsize=(12, 2.8 * len(top)), sharex=True)
if len(top) == 1: axes = [axes]

for i, (ax, c) in enumerate(zip(axes, top)):
    color = cmap(i)
    ax.fill_between(exp_tt, exp_I, alpha=0.2, color='grey')
    ax.plot(exp_tt, exp_I, lw=0.6, color='grey', label='experimental')

    # Tick marks below axis
    for tt_p, I_p in zip(np.array(c.simulated_pattern.x), np.array(c.simulated_pattern.y) / 100):
        ax.plot([tt_p, tt_p], [0, -0.15 * I_p], color=color, lw=1.2)

    # Broadened overlay
    sim_b = broadened_pattern(c, exp_tt)
    if sim_b.max() > 0: sim_b /= sim_b.max()
    ax.plot(exp_tt, sim_b, color=color, lw=1.2, alpha=0.85,
            label=f"{c.formula}  ({c.space_group})  score={c.score:.3f}")

    ax.set_ylim(-0.25, 1.2); ax.set_ylabel('I (norm.)')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_title(f"#{i+1}  {c.mp_id}  Ehull={c.energy_above_hull:.4f} eV/at", fontsize=9)

axes[-1].set_xlabel('2θ (degrees)')
plt.suptitle('Top candidates vs. experimental pattern', fontsize=11, y=1.01)
plt.tight_layout(); plt.show()

---
## 10. Peak Map — Experimental vs. Single Candidate

Change `CANDIDATE_IDX` to inspect any ranked candidate.

In [ ]:
CANDIDATE_IDX = 0

c = ranked[CANDIDATE_IDX]
exp_pk_tt, exp_pk_I = extract_peaks(processed)
sim_tt = np.array(c.simulated_pattern.x)
sim_I  = np.array(c.simulated_pattern.y) / 100.0

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(processed.two_theta, processed.intensity / processed.intensity.max(), lw=0.7, color='k', label='experimental')
ax.vlines(exp_pk_tt,  0,  exp_pk_I,          color='green',  lw=1.5, alpha=0.7, label='exp. peaks')
ax.vlines(sim_tt,    -0.05, -0.05 - 0.18*sim_I, color='crimson', lw=1.5, alpha=0.8, label=f'{c.formula} sim.')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('2θ (degrees)'); ax.set_ylabel('Intensity (norm.)')
ax.set_title(f'Peak map — {c.formula}  |  {c.space_group}  |  score={c.score:.3f}')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"Experimental peaks : {len(exp_pk_tt)}")
print(f"Simulated peaks    : {len(sim_tt)}")

---
## ✅  CHECKPOINT 1 — Phase Hypothesis Selection

Review the overlays above, then fill in `SELECTED_INDICES` below.

**Questions to ask yourself**:
- Do simulated peak positions match experimental peaks?
- Are there unmatched experimental peaks → additional phase?
- Is this phase plausible under your synthesis conditions?
- Are there isostructural alternatives (same SG#) worth testing?

Your selection is saved to `cp1_selection.json` and used by notebook 02 (Rietveld).

In [ ]:
# ── HUMAN INPUT — edit this ──────────────────────────────────────────────────
SELECTED_INDICES = [0]   # 0-based rank indices, e.g. [0] or [0, 2]
NOTES = ""               # optional free-text
# ────────────────────────────────────────────────────────────────────────────

selected = [ranked[i] for i in SELECTED_INDICES]
print(f"Selected {len(selected)} phase(s) for Rietveld:")
for i, c in zip(SELECTED_INDICES, selected):
    print(f"  [{i}] {c.mp_id}  {c.formula}  SG#{c.space_group_number}  score={c.score:.3f}")

In [ ]:
cp_dir = Path("checkpoints")
cp_dir.mkdir(exist_ok=True)

cp1_data = {
    "xrd_file": str(XRD_FILE),
    "elements": ELEMENTS,
    "wavelength": WAVELENGTH,
    "scorer": "dara" if bgmn_available else "pymatgen",
    "notes": NOTES,
    "selected_phases": [
        {"mp_id": c.mp_id, "formula": c.formula, "space_group": c.space_group,
         "space_group_number": c.space_group_number,
         "energy_above_hull": c.energy_above_hull, "score": c.score}
        for c in selected
    ],
}

with open(cp_dir / "cp1_selection.json", "w") as fh:
    json.dump(cp1_data, fh, indent=2)
with open(cp_dir / "cp1_selected_candidates.pkl", "wb") as fh:
    pickle.dump({"pattern": processed, "selected": selected}, fh)

print("✅  Checkpoint 1 saved:")
print(f"   {cp_dir / 'cp1_selection.json'}")
print(f"   {cp_dir / 'cp1_selected_candidates.pkl'}")
print("\nNext → notebook_02_rietveld.ipynb")